# Part 1: Code Q&A System with Bash Tools

Answers questions about the `mcp-gateway-registry` codebase using bash tools for context retrieval.

**Architecture:** User Query → Query Router → Bash Tools → LLM Answer

All pipeline logic lives in `src/part1_pipeline.py`.

## Step 0: Clone the Repository

In [5]:
# Clone the mcp-gateway-registry codebase (run only once)
import os

if not os.path.exists("./mcp-gateway-registry"):
    os.system("git clone https://github.com/agentic-community/mcp-gateway-registry")
    print("Repository cloned successfully.")
else:
    print("Repository already exists, skipping clone.")


Repository already exists, skipping clone.


## Step 1: Setup

In [6]:
import sys
import os

# Make sure the project root is on the path so src/ can be imported
sys.path.insert(0, os.path.abspath(".."))

from src.part1_pipeline import answer_question, classify_query
from src.config import REPO_PATH, GROQ_MODEL

print("Imports OK")
print(f"  Repo  : {REPO_PATH}")
print(f"  Model : {GROQ_MODEL}")


Imports OK
  Repo  : ./mcp-gateway-registry
  Model : llama-3.3-70b-versatile


## Step 2: Verify the Router

Quick sanity check — confirm the router correctly classifies a simple question.

In [7]:
import json

# Test router on one question before running all 6
test = classify_query("What Python dependencies does this project use?")
print(json.dumps(test, indent=2))


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kesphj16f2q8j3nbn107y3qz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99990, Requested 723. Please try again in 10m16.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Step 3: Questions 1–3 (Simple)

Dependency lookup, entry point, and file structure — answered with `cat` and `tree`.

In [ ]:
simple_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
]

RESULTS_FILE = "./part1_results.txt"

# Start results file fresh
with open(RESULTS_FILE, "w") as f:
    f.write("Part 1: Code Q&A System Results\n" + "=" * 60 + "\n\n")

for i, question in enumerate(simple_questions, 1):
    answer = answer_question(question)
    print(f"\n{'='*60}\n[ANSWER]\n{'='*60}")
    print(answer)
    # Save immediately after each answer
    with open(RESULTS_FILE, "a") as f:
        f.write("Q" + str(i) + ": " + question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: What Python dependencies does this project use?

[Retrieving context...]
[Router] Query type : dependency
[Router] Reasoning  : Read pyproject.toml to find declared Python dependencies.
[Router] Executing 2 command(s):
  1. cat ./mcp-gateway-registry/pyproject.toml
  2. grep --include='*.py' -r 'import' ./mcp-gateway-registry | head -20

[Generating answer...]

[ANSWER]
The project uses a variety of Python dependencies, which are specified in the `pyproject.toml` file. 

The dependencies are listed under the `[project.dependencies]` section. Some of the key dependencies include:
```toml
dependencies = [
    "aiofiles>=24.1.0",
    "fastapi>=0.115.12",
    "itsdangerous>=2.2.0",
    "jinja2>=3.1.6",
    "mcp>=1.9.3",
    "pydantic>=2.11.3",
    "pydantic-settings>=2.0.0",
    "httpx>=0.27.0",
    "python-dotenv>=1.1.0",
    "python-multipart>=0.0.20",
    "uvicorn[standard]>=0.34.2",
    "faiss-cpu>=1.7.4",
    "sentence-transformers>=3.0.0",
    "websockets>=15.0.1",
    "sc

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kesphj16f2q8j3nbn107y3qz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 744. Please try again in 10m42.816s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Step 4: Questions 4–5 (Difficult)

Multi-file search using `grep` + `cat` combinations.

In [ ]:
difficult_questions = [
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
]

for i, question in enumerate(difficult_questions, 4):
    answer = answer_question(question)
    print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
    print(answer)
    with open("./part1_results.txt", "a") as f:
        f.write("Q" + str(i) + ": " + question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
    print("[Saved Q" + str(i) + "]")


Question: How does the authentication flow work, from token validation to user authorization?

[Retrieving context...]
[Router] Query type : multi
[Router] Reasoning  : The question requires a combination of searching for keywords in code and documentation to understand the authentication flow.
[Router] Executing 5 command(s):
  1. grep -r --include='*.py' 'token validation' ./mcp-gateway-registry
  2. grep -r --include='*.md' 'authentication flow' ./mcp-gateway-registry/docs
  3. cat $(grep -r --include='*.py' 'token validation' ./mcp-gateway-registry | cut -d: -f1)
  4. cat $(grep -r --include='*.md' 'authentication flow' ./mcp-gateway-registry/docs | cut -d: -f1)
  5. grep -r --include='*.py' 'user authorization' ./mcp-gateway-registry

[Generating answer...]

[ANSWER]
Based on the provided codebase context, I'll outline the authentication flow from token validation to user authorization. Please note that some parts of the flow might be uncertain due to the limited context.

The au

## Step 5: Question 6 (Very Hard)

Synthesizes code, docs, and config across multiple directories.

In [ ]:
hard_question = (
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? "
    "What files would need to be modified and what interfaces must be implemented?"
)

answer = answer_question(hard_question)
print("\n" + "="*60 + "\n[ANSWER]\n" + "="*60)
print(answer)
with open("./part1_results.txt", "a") as f:
    f.write("Q6: " + hard_question + "\n" + "=" * 60 + "\n" + answer + "\n\n")
print("[Saved Q6]")
print("\n✅ All results saved to part1_results.txt")


Question: How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?

[Retrieving context...]
[Router] Query type : multi
[Router] Reasoning  : To add support for a new OAuth provider, we need to search for relevant code and documentation, then inspect the contents of those files.
[Router] Executing 5 command(s):
  1. grep -r --include='*.py' 'oauth' ./mcp-gateway-registry
  2. grep -r --include='*.md' 'oauth' ./mcp-gateway-registry/docs
  3. cat ./mcp-gateway-registry/auth.py
  4. cat ./mcp-gateway-registry/docs/authentication.md
  5. find ./mcp-gateway-registry -name 'oauth*' -print

[Generating answer...]

[ANSWER]
To add support for a new OAuth provider, such as Okta, to the authentication system, you would need to modify the following files and implement the required interfaces:

1. **`credentials-provider/oauth/oauth_providers.yaml`**: This file contains provider-spec

## (Optional) Interactive Mode

In [ ]:
# Uncomment to ask your own questions
# while True:
#     q = input("Ask a question (or exit): ").strip()
#     if q.lower() in ("exit", "quit", ""):
#         break
#     print(answer_question(q))
